In [22]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [23]:
vocab_size = 10000
maxlen = 200
batch_size = 32

In [24]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

In [25]:
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

In [26]:
x_train_torch = torch.LongTensor(x_train)
y_train_torch = torch.FloatTensor(y_train).view(-1, 1)

x_test_torch = torch.LongTensor(x_test)
y_test_torch = torch.FloatTensor(y_test).view(-1, 1)

In [27]:
train_ds = TensorDataset(x_train_torch, y_train_torch)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

test_ds = TensorDataset(x_test_torch, y_test_torch)
test_loader = DataLoader(test_ds, batch_size=batch_size)

In [28]:
class IMDB_LSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(IMDB_LSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embed = self.embedding(x)
        _, (hidden, _) = self.lstm(embed)

        out = self.fc(hidden[-1])
        return self.sigmoid(out)

In [29]:
class IMDB_RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embed = self.embedding(x)
        _, hidden = self.rnn(embed)
        out = self.fc(hidden[-1])
        return self.sigmoid(out)

In [30]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = IMDB_LSTM(vocab_size, 128, 64).to(device)
rnn_model = IMDB_RNN(vocab_size, 128, 64).to(device)

In [31]:
criterion = nn.BCELoss()
optimizer_l = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
optimizer_r = torch.optim.Adam(rnn_model.parameters(), lr=0.001)

In [32]:
def train_model(model, optimizer, train_loader, epochs):
    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

In [34]:
lm=train_model(lstm_model,optimizer_l,train_loader,5)

Epoch [1/5], Loss: 0.5554
Epoch [2/5], Loss: 0.4380
Epoch [3/5], Loss: 0.3369
Epoch [4/5], Loss: 0.2817
Epoch [5/5], Loss: 0.2139


In [35]:
rnn=train_model(rnn_model,optimizer_r,train_loader,5)

Epoch [1/5], Loss: 0.6683
Epoch [2/5], Loss: 0.6138
Epoch [3/5], Loss: 0.5349
Epoch [4/5], Loss: 0.4693
Epoch [5/5], Loss: 0.4192


In [43]:
def model_eval(model,test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for texts, labels in test_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return (f"{100 * correct / total}%")

In [44]:
ml=model_eval(lstm_model,test_loader)
mr=model_eval(rnn_model,test_loader)

In [74]:
print(" ___________________")
print("| Model  |  Score   |")
print("|--------|----------|")
print(f"| LSTM   |  {ml} |")
print(f"| RNN    |  {mr} |")
print("|________|__________|")

 ___________________
| Model  |  Score   |
|--------|----------|
| LSTM   |  83.788% |
| RNN    |  76.688% |
|________|__________|
